In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{mean squared error} $$

$$ \mathbb{R}^n \times \mathbb{R}^n \to \mathbb{R}, \quad y(\mathbf{p}, \mathbf{t}) = \frac{1}{N} || \mathbf{p} - \mathbf{t} ||^2 $$

$$ \frac{\partial \mathbf{y}}{\partial \mathbf{p}} = \frac{2}{N} (\mathbf{p} - \mathbf{t}) $$

$$
J_y(\mathbf{p}) =
\nabla _y(\mathbf{p}) \top =
\begin{bmatrix}
    \frac{\partial y}{\partial p_1}, &
    \frac{\partial y}{\partial p_2},  &
    \cdots, &
    \frac{\partial y}{\partial p_n}
\end{bmatrix}
$$

$$ d\mathbf{y} = J_y(\mathbf{p}) \cdot d\mathbf{p} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dp}, d\mathbf{p} \right \rangle =
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle
$$

$$ \\[2em] $$

$$
\left \langle \frac{dF}{dy}, d\mathbf{y} \right \rangle =
\left \langle \frac{dF}{dy}, J_y(\mathbf{p}) \, d\mathbf{p} \right \rangle =
\left \langle J_y(\mathbf{p}) ^\top \, \frac{dF}{dy}, d\mathbf{p} \right \rangle \implies
$$

$$ \frac{dF}{dp} = J_y(\mathbf{p})^\top \, \frac{dF}{dy} = \frac{dy}{dp} \odot \frac{dF}{dy} $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def mse(p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
    """Computes the mean squared error between predictions `p` and targets `t`."""
 
    return ((p - t) ** 2).mean()


class MeanSquaredErrorFunction(autograd.Function):
    """Custom autograd function for mean squared error."""

    @staticmethod
    def forward(ctx, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        ctx.save_for_backward(p, t)
        return mse(p, t)
    
    @staticmethod
    def backward(ctx, dF_dy: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor]:
        (p, t) = ctx.saved_tensors
        S = p.shape[0]
        FO = p.shape[1]
        N = S * FO

        dy_dp = 2 / N * (p - t)
        dF_dp = dy_dp * dF_dy
        
        return (dF_dp, None)
    

class MeanSquaredError(nn.Module):
    """Mean squared error loss module."""

    def __init__(self):
        super().__init__()
    
    def forward(self, p: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        L = MeanSquaredErrorFunction.apply(p, t)
        return L
    

def test_basic():
  p = tr.tensor([0.2, -1.0, 3.0, 2.5], dtype=tr.float32)
  t = tr.tensor([0.0, -2.0, 1.0, 2.0], dtype=tr.float32)

  actual = mse(p, t)
  expected = tr.nn.MSELoss()(p, t)

  assert actual == approx(expected)


def test_gradcheck(samples) -> None:
    def fn(p, t):
        return MeanSquaredErrorFunction.apply(p, t)

    p = tr.randn((samples, 1), dtype=tr.float64, requires_grad=True)
    t = tr.randn((samples, 1), dtype=tr.float64, requires_grad=False)

    assert autograd.gradcheck(fn, (p, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    p = tr.randn(samples, dtype=tr.float32, requires_grad=True)
    t = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    mse_custom = MeanSquaredErrorFunction.apply(p, t)
    mse_builtin = tr.nn.MSELoss()(p, t)

    assert mse_custom == approx(mse_builtin)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)
